# Gas Transport Physics-Informed GPR

This notebook models polymer gas permeability using the solution-diffusion relationship. The first target is experimental CO2 permeability for rows where experimental diffusivity and solubility are also available.

## Paper Reference

Phan, B. K.; Shen, K.-H.; Gurnani, R.; Tran, H.; Lively, R.; Ramprasad, R. **Gas permeability, diffusivity, and solubility in polymers: Simulation-experiment data fusion and multi-task machine learning.** npj Computational Materials 10, 185 (2024).

Physics used here:

- Solution-diffusion model: `P = D*S`.
- In log10 units: `logP = logD + logS`.
- The physics-informed GPR variants learn residual corrections around this relation.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "matgpr").exists():
            return candidate
        sibling = candidate / "matgpr"
        if (sibling / "pyproject.toml").exists() and (sibling / "matgpr").exists():
            return sibling
    raise FileNotFoundError("Could not find the local matgpr package root.")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Using package from: {PROJECT_ROOT}")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from gpytorch.utils.warnings import GPInputWarning
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from matgpr import PhysicsInformedMean, featurize_smiles, fit_gpytorch_gpr

RANDOM_STATE = 42
plt.style.use("default")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
torch.set_num_threads(1)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=GPInputWarning)

## 2. Load CO2 Permeability Dataset

In [ ]:
DATA_PATH = NOTEBOOK_DIR / "gas_transport_wide.csv"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "examples" / "gas_transport" / "gas_transport_wide.csv"

TARGET_GAS = "CO2"
TARGET_COLUMN = f"p_exp_{TARGET_GAS}"
DIFFUSIVITY_COLUMN = f"d_exp_{TARGET_GAS}"
SOLUBILITY_COLUMN = f"s_exp_{TARGET_GAS}"

raw_data = pd.read_csv(DATA_PATH)
print(f"Raw rows: {len(raw_data):,}")
raw_data[["smiles_string", TARGET_COLUMN, DIFFUSIVITY_COLUMN, SOLUBILITY_COLUMN]].head()

In [ ]:
data = raw_data[["smiles_string", TARGET_COLUMN, DIFFUSIVITY_COLUMN, SOLUBILITY_COLUMN]].copy()
data = data.rename(
    columns={
        "smiles_string": "polymer_smiles",
        TARGET_COLUMN: "log10_permeability",
        DIFFUSIVITY_COLUMN: "log10_diffusivity",
        SOLUBILITY_COLUMN: "log10_solubility",
    }
)
data = data.dropna().reset_index(drop=True)
data = data[data["polymer_smiles"].astype(str).str.count(r"\[\*\]").eq(2)].reset_index(drop=True)
print(f"Rows with P, D, S and exactly two [*] markers: {len(data):,}")
data.head()

## 3. Polymer Fingerprints

Polymer repeat units must contain exactly two `[*]` atoms. For fingerprinting, each repeat unit is converted into a cyclic trimer surrogate: adjacent units are connected through the two dummy-end neighbors using the dummy-end bond order, the trimer is closed into a loop, all `[*]` atoms are removed, and the resulting molecule is RDKit-canonicalized before Morgan fingerprints and descriptors are computed.

In [ ]:
FINGERPRINT_BITS = 128
polymer_result = featurize_smiles(
    data["polymer_smiles"],
    smiles_type="polymer",
    fingerprint_type="morgan+descriptors",
    n_bits=FINGERPRINT_BITS,
    column_prefix="polymer",
    errors="coerce",
)
model_data = pd.concat([data.reset_index(drop=True), polymer_result.canonical_smiles, polymer_result.features], axis=1)
model_data = model_data.dropna().reset_index(drop=True)
print(f"Rows after RDKit featurization: {len(model_data):,}")
model_data[["polymer_canonical_smiles", "log10_permeability", "log10_diffusivity", "log10_solubility"]].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.4))
for ax, column, title, color in [
    (axes[0], "log10_permeability", "Permeability", "#2f6f8f"),
    (axes[1], "log10_diffusivity", "Diffusivity", "#4a7c59"),
    (axes[2], "log10_solubility", "Solubility", "#b85c38"),
]:
    ax.hist(model_data[column], bins=25, color=color, alpha=0.9)
    ax.set_xlabel(column)
    ax.set_title(title)
axes[0].set_ylabel("Count")
fig.tight_layout()

## 4. Solution-Diffusion Mean Functions

In [ ]:
PHYSICS_FEATURES = ["log10_diffusivity", "log10_solubility"]
FINGERPRINT_COLUMNS = [column for column in model_data.columns if column.startswith("polymer_morgan_") or column.startswith("polymer_desc_")]
FEATURE_COLUMNS = FINGERPRINT_COLUMNS + PHYSICS_FEATURES
TARGET_COLUMN = "log10_permeability"

MODEL_CONFIGS = {
    "standard_gpr": {"label": "Standard GPR", "physics": "none"},
    "pi_solution_fixed": {"label": "PI-GPR: logD + logS", "physics": "fixed"},
    "pi_solution_learned": {"label": "PI-GPR: learned D/S weights", "physics": "learned"},
    "pi_diffusivity": {"label": "PI-GPR: diffusivity", "physics": "diffusivity"},
    "pi_solubility": {"label": "PI-GPR: solubility", "physics": "solubility"},
}
MODEL_ORDER = list(MODEL_CONFIGS)
MODEL_LABELS = {key: value["label"] for key, value in MODEL_CONFIGS.items()}
MODEL_COLORS = {
    "standard_gpr": "#4c5a72",
    "pi_solution_fixed": "#2f6f8f",
    "pi_solution_learned": "#b85c38",
    "pi_diffusivity": "#4a7c59",
    "pi_solubility": "#8064a2",
}

TRAINING_PERCENTS = np.arange(10, 101, 10)
N_RANDOM_SPLITS = 20
TEST_SIZE = 0.20
TRAINING_ITER = 120
PRODUCTION_TRAINING_ITER = 250

In [ ]:
def solution_fixed_mean(features, parameters):
    return parameters["bias"] + features["log10_diffusivity"] + features["log10_solubility"]


def solution_learned_mean(features, parameters):
    return parameters["bias"] + parameters["w_d"] * features["log10_diffusivity"] + parameters["w_s"] * features["log10_solubility"]


def diffusivity_mean(features, parameters):
    return parameters["bias"] + parameters["w_d"] * features["log10_diffusivity"]


def solubility_mean(features, parameters):
    return parameters["bias"] + parameters["w_s"] * features["log10_solubility"]


def build_physics_mean(model_key, feature_columns, scaler, y_train):
    physics = MODEL_CONFIGS[model_key]["physics"]
    if physics == "none":
        return None
    feature_means = dict(zip(feature_columns, scaler.mean_))
    feature_stds = dict(zip(feature_columns, scaler.scale_))
    common = {
        "feature_indices": {name: feature_columns.index(name) for name in PHYSICS_FEATURES},
        "feature_means": {name: feature_means[name] for name in PHYSICS_FEATURES},
        "feature_stds": {name: feature_stds[name] for name in PHYSICS_FEATURES},
    }
    bias0 = float(np.median(y_train - (feature_means["log10_diffusivity"] + feature_means["log10_solubility"])))
    if physics == "fixed":
        return PhysicsInformedMean(equation=solution_fixed_mean, learnable_parameters={"bias": bias0}, **common)
    if physics == "learned":
        return PhysicsInformedMean(
            equation=solution_learned_mean,
            learnable_parameters={"bias": bias0, "w_d": 1.0, "w_s": 1.0},
            positive_parameters=("w_d", "w_s"),
            **common,
        )
    if physics == "diffusivity":
        return PhysicsInformedMean(equation=diffusivity_mean, learnable_parameters={"bias": bias0, "w_d": 1.0}, positive_parameters=("w_d",), **common)
    if physics == "solubility":
        return PhysicsInformedMean(equation=solubility_mean, learnable_parameters={"bias": bias0, "w_s": 1.0}, positive_parameters=("w_s",), **common)
    raise ValueError(f"Unknown physics model: {physics}")


def fit_model(model_key, train_frame, *, training_iter=TRAINING_ITER):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_frame[FEATURE_COLUMNS])
    y_train = train_frame[TARGET_COLUMN].to_numpy(dtype=float)
    mean_module = build_physics_mean(model_key, FEATURE_COLUMNS, scaler, y_train)
    result = fit_gpytorch_gpr(
        X_train,
        y_train,
        kernel="matern",
        ard=False,
        mean_module=mean_module,
        lr=0.03,
        training_iter=training_iter,
        initial_noise=0.05,
        standardize_y=True,
        verbose=False,
    )
    return {"model_key": model_key, "scaler": scaler, "result": result}


def predict_model(fitted_model, frame, *, confidence_level=0.95):
    X = fitted_model["scaler"].transform(frame[FEATURE_COLUMNS])
    return fitted_model["result"].predict(X, confidence_level=confidence_level)

## 5. Learning Curves

In [ ]:
def regression_summary(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    r_value = pearsonr(y_true, y_pred).statistic if len(np.unique(y_true)) > 1 else np.nan
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "r": float(r_value),
    }


def target_strata(frame, target_column, bins=8):
    return pd.qcut(frame[target_column], q=bins, labels=False, duplicates="drop")


def subset_training_pool(train_pool, target_column, training_percent, *, random_state):
    if training_percent >= 100:
        return train_pool.copy().reset_index(drop=True)
    train_size = training_percent / 100
    strata = target_strata(train_pool, target_column)
    try:
        subset, _ = train_test_split(
            train_pool,
            train_size=train_size,
            random_state=random_state,
            stratify=strata,
        )
    except ValueError:
        subset = train_pool.sample(frac=train_size, random_state=random_state)
    return subset.reset_index(drop=True)


def plot_learning_curves(learning_summary, model_order, model_labels, model_colors):
    fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.1), sharex=True)
    for model_key in model_order:
        subset = learning_summary[learning_summary["model_key"] == model_key]
        color = model_colors.get(model_key)
        axes[0].errorbar(
            subset["training_percent"],
            subset["rmse_mean"],
            yerr=subset["rmse_std"],
            marker="o",
            linewidth=2.0,
            capsize=3,
            label=model_labels[model_key],
            color=color,
        )
        axes[1].errorbar(
            subset["training_percent"],
            subset["r2_mean"],
            yerr=subset["r2_std"],
            marker="o",
            linewidth=2.0,
            capsize=3,
            label=model_labels[model_key],
            color=color,
        )
    axes[0].set_ylabel("Held-out RMSE")
    axes[1].set_ylabel("Held-out R2")
    for ax in axes:
        ax.set_xlabel("Training pool used (%)")
        ax.set_xticks(TRAINING_PERCENTS)
        ax.legend(frameon=False)
    fig.tight_layout()
    return fig, axes

In [ ]:
learning_records = []

for repeat in range(N_RANDOM_SPLITS):
    split_seed = RANDOM_STATE + repeat
    train_pool, test_set = train_test_split(
        model_data,
        test_size=TEST_SIZE,
        random_state=split_seed,
        stratify=target_strata(model_data, TARGET_COLUMN, bins=5),
    )
    train_pool = train_pool.reset_index(drop=True)
    test_set = test_set.reset_index(drop=True)

    for training_percent in TRAINING_PERCENTS:
        subset = subset_training_pool(train_pool, TARGET_COLUMN, int(training_percent), random_state=split_seed + int(training_percent) * 100)
        for model_key in MODEL_ORDER:
            fitted = fit_model(model_key, subset, training_iter=TRAINING_ITER)
            prediction = predict_model(fitted, test_set)
            metrics = regression_summary(test_set[TARGET_COLUMN], prediction.mean)
            learning_records.append({"repeat": repeat, "training_percent": int(training_percent), "n_train": len(subset), "model_key": model_key, "model": MODEL_LABELS[model_key], **metrics})

learning_results = pd.DataFrame(learning_records)
learning_summary = (
    learning_results
    .groupby(["training_percent", "model_key", "model"], as_index=False)
    .agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), r2_mean=("r2", "mean"), r2_std=("r2", "std"), mae_mean=("mae", "mean"), mae_std=("mae", "std"))
)
learning_summary.head()

In [ ]:
plot_learning_curves(learning_summary, MODEL_ORDER, MODEL_LABELS, MODEL_COLORS)

## 6. Low-Data Selection and Parity

In [ ]:
SELECTION_PERCENT = 20
selection_table = learning_summary[learning_summary["training_percent"] == SELECTION_PERCENT].sort_values(["rmse_mean", "rmse_std"]).reset_index(drop=True)
best_model_key = selection_table.loc[0, "model_key"]
best_model_label = MODEL_LABELS[best_model_key]
print(f"Selected low-data model: {best_model_label}")
selection_table[["model", "rmse_mean", "rmse_std", "r2_mean", "r2_std"]]

In [ ]:
train_pool, test_set = train_test_split(model_data, test_size=TEST_SIZE, random_state=RANDOM_STATE + 999, stratify=target_strata(model_data, TARGET_COLUMN, bins=5))
low_data_train = subset_training_pool(train_pool.reset_index(drop=True), TARGET_COLUMN, SELECTION_PERCENT, random_state=RANDOM_STATE + 123)
standard_model = fit_model("standard_gpr", low_data_train, training_iter=PRODUCTION_TRAINING_ITER)
best_model = fit_model(best_model_key, low_data_train, training_iter=PRODUCTION_TRAINING_ITER)
standard_pred = predict_model(standard_model, test_set)
best_pred = predict_model(best_model, test_set)
standard_metrics = regression_summary(test_set[TARGET_COLUMN], standard_pred.mean)
best_metrics = regression_summary(test_set[TARGET_COLUMN], best_pred.mean)

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.7), sharex=True, sharey=True)
for ax, label, pred, metrics, color in [(axes[0], "Standard GPR", standard_pred, standard_metrics, MODEL_COLORS["standard_gpr"]), (axes[1], best_model_label, best_pred, best_metrics, MODEL_COLORS.get(best_model_key, "#b85c38"))]:
    ax.errorbar(test_set[TARGET_COLUMN], pred.mean, yerr=pred.std, fmt="o", ms=4, alpha=0.75, capsize=1.8, color=color)
    limits = [test_set[TARGET_COLUMN].min(), test_set[TARGET_COLUMN].max()]
    pad = 0.05 * (limits[1] - limits[0])
    limits = [limits[0] - pad, limits[1] + pad]
    ax.plot(limits, limits, "--", color="#2b2b2b", linewidth=1)
    ax.set_xlim(limits)
    ax.set_ylim(limits)
    ax.set_title(label)
    ax.set_xlabel("Observed log10 permeability")
    ax.text(0.04, 0.96, f"RMSE={metrics['rmse']:.3f}\\nR2={metrics['r2']:.3f}\\nr={metrics['r']:.3f}", transform=ax.transAxes, va="top", fontsize=9)
axes[0].set_ylabel("Predicted log10 permeability")
fig.tight_layout()

In [ ]:
production_model = fit_model(best_model_key, model_data, training_iter=PRODUCTION_TRAINING_ITER)
production_prediction = predict_model(production_model, model_data)
production_metrics = regression_summary(model_data[TARGET_COLUMN], production_prediction.mean)
print(f"Production model: {best_model_label}")
print(production_metrics)
mean_module = production_model["result"].model.mean_module
if hasattr(mean_module, "current_parameter_values"):
    print(mean_module.current_parameter_values())